# Session Integration

> Schema-knowledge helpers for presenting decomposition workflow sessions in a `cjm-fasthtml-workflow-session-management` session manager.

These helpers know the shape of `StructureDecompWorkflow`'s persisted state and translate it into the display fields the session manager UI needs:

- `DECOMP_STEP_TITLES` + `get_decomp_step_title` — map internal step IDs to human titles
- `decomp_enricher` — count selected sources and segments from a raw state blob
- `decomp_label_generator` — derive a meaningful default label from the first selected source's media path

Hosts wiring up session management over this workflow import and pass these directly to `init_session_manager_routers(...)`.

In [ ]:
#| default_exp workflow.session_integration

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from pathlib import Path
from typing import Any, Dict

from cjm_workflow_state.state_store import SessionSummary
from cjm_fasthtml_workflow_session_management.utils import default_label

## Step Titles

StepFlow stores raw step IDs in the state store (e.g., `"decomposition"`, `"verify"`). The session manager list displays a "Step" column that should show human-readable titles instead. `DECOMP_STEP_TITLES` is the mapping; `get_decomp_step_title` is a safe lookup that falls back to the raw ID when a title isn't defined.

Note that `"decomposition"` and `"segmentation"` both map to `"Segment & Align"` — the step ID was never renamed to avoid breaking existing state, but the user-facing title changed.

In [ ]:
#| export
DECOMP_STEP_TITLES:Dict[str, str] = {
    "selection": "Selection",
    "decomposition": "Segment & Align",
    "segmentation": "Segment & Align",  # alias for any state persisted under the old ID
    "review": "Review",
    "verify": "Verify",
}

def get_decomp_step_title(
    step_id:str # Raw step ID from the state store
) -> str: # Human-readable step title, or the raw ID if not mapped
    """Look up the human-readable title for a decomposition workflow step ID."""
    if not step_id:
        return ""
    return DECOMP_STEP_TITLES.get(step_id, step_id)

In [ ]:
assert get_decomp_step_title("selection") == "Selection"
assert get_decomp_step_title("decomposition") == "Segment & Align"
assert get_decomp_step_title("segmentation") == "Segment & Align"
assert get_decomp_step_title("verify") == "Verify"
# Unknown IDs fall through to the raw string.
assert get_decomp_step_title("custom") == "custom"
# Empty/None-ish inputs return empty.
assert get_decomp_step_title("") == ""
print("Step title lookup OK")

Step title lookup OK


## Enricher

`decomp_enricher` reads the raw `state_json` dict that `SessionManagementService` passes in, and returns display strings for the "Sources" and "Segments" columns. It reaches into:

- `state_json["step_states"]["selection"]["selected_sources"]` — a list of selected source records
- `state_json["step_states"]["segmentation"]["segments"]` **or** `state_json["step_states"]["decomposition"]["segments"]` — the text segment list (both keys supported for forward/backward compatibility)

If a field is missing or empty, the corresponding cell shows `"—"` rather than `"0"` — this visually distinguishes "not yet populated" from "populated but empty".

In [ ]:
#| export
def decomp_enricher(
    state_json:Dict[str, Any] # Raw session state blob from the state store
) -> Dict[str, str]: # Display fields keyed by column name
    """Extract source count and segment count from decomposition workflow state."""
    step_states = state_json.get("step_states", {}) or {}
    
    selection_state = step_states.get("selection", {}) or {}
    sources = selection_state.get("selected_sources", []) or []
    
    # Segments may live under either key depending on when the state was persisted.
    seg_state = (
        step_states.get("segmentation", {})
        or step_states.get("decomposition", {})
        or {}
    )
    segments = seg_state.get("segments", []) or []
    
    return {
        "sources": str(len(sources)) if sources else "—",
        "segments": str(len(segments)) if segments else "—",
    }

In [ ]:
# Enricher dispatch + defensive handling of missing / empty / malformed input.

# Empty state → placeholders.
assert decomp_enricher({}) == {"sources": "—", "segments": "—"}

# Only sources populated.
sources_only = {
    "step_states": {
        "selection": {"selected_sources": [{"record_id": "a"}, {"record_id": "b"}]}
    }
}
assert decomp_enricher(sources_only) == {"sources": "2", "segments": "—"}

# Both populated under the canonical "segmentation" key.
both = {
    "step_states": {
        "selection": {"selected_sources": [{"record_id": "x"}]},
        "segmentation": {"segments": [1, 2, 3, 4, 5]},
    }
}
assert decomp_enricher(both) == {"sources": "1", "segments": "5"}

# Backwards-compat: segments stored under the old "decomposition" key.
legacy = {
    "step_states": {
        "selection": {"selected_sources": [{"record_id": "x"}]},
        "decomposition": {"segments": [1, 2, 3]},
    }
}
assert decomp_enricher(legacy) == {"sources": "1", "segments": "3"}

# If both keys exist, "segmentation" wins (it's the current/canonical key).
conflicting = {
    "step_states": {
        "segmentation": {"segments": [1, 2]},
        "decomposition": {"segments": [1, 2, 3, 4]},
    }
}
assert decomp_enricher(conflicting) == {"sources": "—", "segments": "2"}

# Malformed step_states (None values) shouldn't crash.
malformed = {"step_states": None}
assert decomp_enricher(malformed) == {"sources": "—", "segments": "—"}

# Missing step_states entirely.
missing = {"other_field": "unrelated"}
assert decomp_enricher(missing) == {"sources": "—", "segments": "—"}

print("Enricher OK")

Enricher OK


## Label Generator

`decomp_label_generator` produces a default label for a session that has no stored label. It reaches into the same `selected_sources` list as the enricher, but instead of counting, it pulls the first source's `record_id` as a compact identifier.

Session state stores sources as `{record_id, provider_id}` pairs — richer metadata like media file paths lives in federated source databases and isn't loaded into workflow state. So the best we can produce here is the `record_id` itself. Users are expected to rename sessions via the manager UI for friendlier labels.

Falls through to the session management library's `default_label()` (a timestamp-based label) when selection hasn't happened yet.

In [ ]:
#| export
def decomp_label_generator(
    summary:SessionSummary, # Session metadata from the state store
    state_json:Dict[str, Any] # Raw session state blob
) -> str: # Resolved display label
    """Derive a default session label from the first selected source, or a timestamp."""
    step_states = state_json.get("step_states", {}) or {}
    selection_state = step_states.get("selection", {}) or {}
    sources = selection_state.get("selected_sources", []) or []
    
    if sources:
        first = sources[0] if isinstance(sources[0], dict) else {}
        record_id = first.get("record_id", "")
        if record_id:
            n = len(sources)
            if n == 1:
                return record_id
            return f"{record_id} (+{n - 1} more)"
    
    return default_label(summary.created_at)

In [ ]:
# Label generator across common cases.
_empty_summary = SessionSummary(
    flow_id="structure_decomposition", session_id="abc", label=None,
    current_step=None, created_at="2026-04-08 10:00:00",
    updated_at="2026-04-08 10:00:00", state_size_bytes=0,
)

# No sources → timestamp fallback.
assert decomp_label_generator(_empty_summary, {}).startswith("Session ")

# Single source → record_id directly.
one_source = {
    "step_states": {
        "selection": {"selected_sources": [
            {"record_id": "job_1963f984", "provider_id": "external:/foo"}
        ]}
    }
}
assert decomp_label_generator(_empty_summary, one_source) == "job_1963f984"

# Multiple sources → "record_id (+N more)".
three_sources = {
    "step_states": {
        "selection": {"selected_sources": [
            {"record_id": "job_A", "provider_id": "p1"},
            {"record_id": "job_B", "provider_id": "p2"},
            {"record_id": "job_C", "provider_id": "p3"},
        ]}
    }
}
assert decomp_label_generator(_empty_summary, three_sources) == "job_A (+2 more)"

# Source present but no record_id → timestamp fallback.
no_record_id = {
    "step_states": {"selection": {"selected_sources": [{"provider_id": "only"}]}}
}
assert decomp_label_generator(_empty_summary, no_record_id).startswith("Session ")

# Malformed source (not a dict) → timestamp fallback.
weird = {"step_states": {"selection": {"selected_sources": ["bad"]}}}
assert decomp_label_generator(_empty_summary, weird).startswith("Session ")

print("Label generator OK")

Label generator OK


## Integration Test: Live State DB

Validates the helpers against the real `configs/workflows/structure_decomp/workflow_state.db` — the live DB the decomp app actually uses. We work on a temp copy so the live DB stays untouched.

Assertions are **structural** only: we don't assume any particular session configuration, just that each session yields a non-crashing enricher result and label. This keeps the test from breaking every time the user interacts with the real app.

In [ ]:
import shutil, tempfile, os
from cjm_workflow_state.state_store import SQLiteWorkflowStateStore

_live_db = Path("../../configs/workflows/structure_decomp/workflow_state.db")
if _live_db.exists():
    _copy = tempfile.NamedTemporaryFile(suffix='.db', delete=False)
    _copy_path = _copy.name
    _copy.close()
    shutil.copy2(_live_db, _copy_path)
    
    store = SQLiteWorkflowStateStore(_copy_path)
    sessions = store.list_sessions("structure_decomposition")
    print(f"Live DB has {len(sessions)} sessions in structure_decomposition")
    
    for s in sessions:
        state = store.get_state("structure_decomposition", s.session_id)
        
        # Enricher: should return a dict with the expected keys and no crashes.
        enriched = decomp_enricher(state)
        assert isinstance(enriched, dict)
        assert "sources" in enriched
        assert "segments" in enriched
        assert isinstance(enriched["sources"], str)
        assert isinstance(enriched["segments"], str)
        
        # Label generator: should return a non-empty string.
        label = decomp_label_generator(s, state)
        assert isinstance(label, str) and label
        
        # Step title: should return a string (empty when current_step is None is OK).
        title = get_decomp_step_title(s.current_step or "")
        assert isinstance(title, str)
        
        print(f"  {s.session_id[:8]}... step={title!r} label={label!r} enriched={enriched}")
    
    os.unlink(_copy_path)
    print("Live DB integration test passed!")
else:
    print(f"(Skipping — {_live_db} not found)")

Live DB has 3 sessions in structure_decomposition
  07b91bcb... step='Segment & Align' label='job_1963f984' enriched={'sources': '1', 'segments': '89'}
  4500a55f... step='Segment & Align' label='job_1963f984' enriched={'sources': '1', 'segments': '89'}
  f25aee0f... step='Verify' label='job_1963f984' enriched={'sources': '1', 'segments': '89'}
Live DB integration test passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()